# Lab 03

## Name: Sebastian Tadeo Quiroz Tejeda

In [49]:
from spark_utils import SparkUtils
su = SparkUtils("spark://spark-master:7077", "Lab03")
su._spark

In [50]:
from spark_utils import SparkUtils

"""
Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,
Product_Category,Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating
"""

columns_types = [
    ("Order_ID", "long"),
    ("Company", "string"),
    ("City", "string"),
    ("Customer_Age", "integer"),
    ("Order_Value", "float"),
    ("Delivery_Time_Min", "float"),
    ("Distance_Km", "float"),
    ("Items_Count", "float"),
    ("Product_Category", "string"),
    ("Payment_Method", "string"),
    ("Customer_Rating", "float"),
    ("Discount_Applied", "float"),
    ("Delivery_Partner_Rating", "float")
]

qcommerce_schema = SparkUtils.generate_schema(columns_types)


In [51]:

qcommerce_df = su._spark \
                .read \
                .option("header", "true") \
                .option("delimiter", ",") \
                .schema(qcommerce_schema) \
                .csv("/opt/spark/work-dir/data/qcommerce/")
qcommerce_df.printSchema()
qcommerce_df.show(5)

root
 |-- Order_ID: long (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Order_Value: float (nullable = true)
 |-- Delivery_Time_Min: float (nullable = true)
 |-- Distance_Km: float (nullable = true)
 |-- Items_Count: float (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Customer_Rating: float (nullable = true)
 |-- Discount_Applied: float (nullable = true)
 |-- Delivery_Partner_Rating: float (nullable = true)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+----------------+----

In [52]:
# Display and count collumns
from pyspark.sql.functions import when, count, isnull
qcommerce_df.columns

['Order_ID',
 'Company',
 'City',
 'Customer_Age',
 'Order_Value',
 'Delivery_Time_Min',
 'Distance_Km',
 'Items_Count',
 'Product_Category',
 'Payment_Method',
 'Customer_Rating',
 'Discount_Applied',
 'Delivery_Partner_Rating']

In [53]:
qcommerce_null_count_df = qcommerce_df.select([count(when(isnull(c), c)).alias(c) for c in qcommerce_df.columns])
qcommerce_null_count_df.show()

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [54]:
qcommerce_null_count_df_2 = SparkUtils.count_nulls(qcommerce_df)
qcommerce_null_count_df_2.show()

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [55]:
qcommerce_wo_null_df = qcommerce_df.dropna()
print(f"Size of dt frame after removing nulls: {qcommerce_wo_null_df.count()}")

Size of dt frame after removing nulls: 780992


In [56]:
qcommerce_wo_nulls_fillna = qcommerce_df.fillna({
    'City': 'Unknown',
    'Items_Count': 0.0,
    'Customer_Rating': 0.0,
    'Delivery_Partner_Rating': 0.0
})
print(f"Size of dt frame after filling nulls: {qcommerce_wo_nulls_fillna.count()}")

Size of dt frame after filling nulls: 1000000


In [57]:
from pyspark.sql.functions import lit
qcommerce_wo_nulls_fillna = qcommerce_df.withColumn("Code_Country", lit("IN"))
qcommerce_wo_nulls_fillna.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.

In [58]:
from pyspark.sql.functions import col
qcommerce_tax_df = qcommerce_wo_nulls_fillna.withColumn("Paid_Tax", col("Order_Value") * 0.16)
qcommerce_tax_df.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|        Paid_Tax|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN| 112.37400390625|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10

## Task 1

In [59]:
task1_df = qcommerce_tax_df.withColumn("Delivery_SLA", 
                                        when(col("Delivery_Time_Min") <= 15, 'FAST').
                                        when(col("Delivery_Time_Min") < 20, 'ON_TIME').
                                        when(col("Delivery_Time_Min") >= 20, 'LATE')) \
                                        .filter(col("Delivery_SLA") == "LATE") \
                                        .orderBy(col("Delivery_Time_Min").desc())

task1_df.select("Order_ID", "Company", "City", "Delivery_Time_Min", "Delivery_SLA").show()

+--------+--------+--------+-----------------+------------+
|Order_ID| Company|    City|Delivery_Time_Min|Delivery_SLA|
+--------+--------+--------+-----------------+------------+
| 1529779|Jio Mart|Haridwar|             40.0|        LATE|
| 1807126|Jio Mart|Haridwar|             40.0|        LATE|
| 1165764|Jio Mart|Haridwar|           39.994|        LATE|
| 1610720|Jio Mart|Haridwar|           39.994|        LATE|
| 1729503|Jio Mart|Haridwar|           39.994|        LATE|
| 1951122|Jio Mart|Haridwar|           39.988|        LATE|
| 1975896|Jio Mart|Haridwar|           39.988|        LATE|
| 1059671|Jio Mart|Haridwar|           39.982|        LATE|
| 1594835|Jio Mart|Haridwar|           39.982|        LATE|
| 1162804|Jio Mart|Haridwar|           39.982|        LATE|
| 1826295|Jio Mart|Haridwar|           39.982|        LATE|
| 1961544|Jio Mart|Haridwar|           39.976|        LATE|
| 1360875|Jio Mart|Haridwar|           39.964|        LATE|
| 1555058|Jio Mart|Haridwar|           3

## Task 2

In [60]:
from pyspark.sql.functions import avg
task2_df = qcommerce_tax_df.withColumn("Effective_Order_Value", col("Order_Value") * (1 - col("Discount_Applied"))). \
                            withColumn("Price_Bucket", 
                                        when(col("Effective_Order_Value") < 200, "LOW").
                                        when(col("Effective_Order_Value") <= 500, "MEDIUM").
                                        when(col("Effective_Order_Value") > 500, "HIGH")). \
                            groupBy("Price_Bucket").agg(
                                count("*").alias("Count"),
                                avg(col("Effective_Order_Value")).alias("Avg_Effective_Order_Value"),
                            ).orderBy(col("Avg_Effective_Order_Value"), ascending=False)
                            

task2_df.show()

+------------+------+-------------------------+
|Price_Bucket| Count|Avg_Effective_Order_Value|
+------------+------+-------------------------+
|        HIGH|268953|        740.4337238893675|
|      MEDIUM|210429|        358.0973233400432|
|         LOW|520618|       21.591204157544553|
+------------+------+-------------------------+



## Task 3

In [61]:
task3_df = qcommerce_tax_df.withColumn("Age_Group", 
                                        when(col("Customer_Age") < 25, "YOUNG").
                                        when(col("Customer_Age") < 45, "ADULT").
                                        when(col("Customer_Age") >= 45, "SENIOR")). \
                            filter(col("Customer_Age") <= 100).\
                            filter(col("Customer_Age") >= 0). \
                            groupBy("Age_Group", "Product_Category").agg(
                                count("*").alias("Orders"),
                                avg(col("Order_Value")).alias("Avg_Order_Value")
                            ). \
                            orderBy("Age_Group"). \
                            orderBy("Orders", ascending=False)

task3_df.show()

+---------+-------------------+------+-----------------+
|Age_Group|   Product_Category|Orders|  Avg_Order_Value|
+---------+-------------------+------+-----------------+
|    ADULT|              Dairy| 68512|573.4268899414931|
|    ADULT|      Personal Care| 68331| 569.512671998805|
|    ADULT|          Groceries| 68291|571.5250993844182|
|    ADULT|          Household| 68110|572.9259149188181|
|    ADULT|             Snacks| 68100|570.3797974095505|
|    ADULT|          Beverages| 67979|572.0329877095578|
|    ADULT|Fruits & Vegetables| 67736|569.8593599596651|
|   SENIOR|          Groceries| 51030|572.2596391052539|
|   SENIOR|              Dairy| 51025| 573.781117268945|
|   SENIOR|             Snacks| 50924| 572.687851608881|
|   SENIOR|          Household| 50803| 571.172082385334|
|   SENIOR|          Beverages| 50746| 568.141013231256|
|   SENIOR|      Personal Care| 50707|571.1642560109325|
|   SENIOR|Fruits & Vegetables| 50642|573.7244422534909|
|    YOUNG|              Dairy|

In [63]:
su._spark.stop()